# EDA 1: competitions and teams (spec sections A and B)

Played matches only; scheduled fixtures are 'Coming soon' and never analysed as results. Source: PostgreSQL warehouse (`docker compose up -d`) and saved reports.

In [1]:
import sys, warnings, pathlib
warnings.filterwarnings("ignore")
ROOT = pathlib.Path.cwd().parent if pathlib.Path.cwd().name == "notebooks" else pathlib.Path.cwd()
sys.path.insert(0, str(ROOT / "src"))
import pandas as pd, numpy as np
import plotly.io as pio
pio.renderers.default = "plotly_mimetype+notebook_connected"
pd.set_option("display.width", 200); pd.set_option("display.max_columns", 30); pd.set_option("display.max_colwidth", 60)


In [2]:
from fbplatform.analysis import eda
res = eda.build()
T, F = res['tables'], res['figures']
print(len(T), 'tables,', len(F), 'figures')

43 tables, 9 figures


## A. Competition level
Per-match rates are used everywhere: raw totals are not comparable across editions with different numbers of matches.

In [3]:
T['A1_competition_summary']

,competition_id,competition_name,seasons,matches_played,goals_per_match,draw_pct,home_win_pct,beyond_90_pct,avg_attendance,goals_stddev,avg_teams,total_goals
0,WC,FIFA World Cup,7,488,2.609,23.0,44.1,8.6,53115.0,1.708,34.285714,1273.0
1,EURO,UEFA European Championship,7,277,2.462,25.3,37.9,9.7,39993.0,1.529,19.428571,682.0
2,COPA,Copa América,9,248,2.512,27.4,46.8,9.7,32088.0,1.740,12.666667,623.0
3,UNL,UEFA Nations League,5,766,2.501,25.2,42.3,0.5,16341.0,1.634,54.400000,1916.0


In [4]:
T['A2_edition_table']

,season_id,competition_id,teams,matches_played,matches_coming_soon,total_goals,goals_per_match,draw_pct,matches_beyond_90,shootouts,avg_attendance,champion,runner_up,tournament_status
0,COPA-2001,COPA,12,26,0,60.0,2.31,15.4,1,1,33731.0,Colombia,Mexico,completed
1,COPA-2004,COPA,12,26,0,78.0,3.00,26.9,2,2,26038.0,Brazil,Argentina,completed
2,COPA-2007,COPA,12,26,0,86.0,3.31,19.2,1,1,38804.0,Brazil,Argentina,completed
3,COPA-2011,COPA,12,26,0,54.0,2.08,42.3,4,3,30796.0,Uruguay,Paraguay,completed
4,COPA-2015,COPA,12,26,0,59.0,2.27,30.8,3,3,25223.0,Chile,Argentina,completed
5,COPA-2016,COPA,16,32,0,91.0,2.84,18.8,2,2,46370.0,Chile,Argentina,completed
6,COPA-2019,COPA,12,26,0,60.0,2.31,34.6,3,3,33356.0,Brazil,Peru,completed
7,COPA-2021,COPA,10,28,0,65.0,2.32,32.1,3,3,279.0,Argentina,Brazil,completed
8,COPA-2024,COPA,16,32,0,70.0,2.19,28.1,5,4,49121.0,Argentina,Colombia,completed
9,EURO-2000,EURO,16,31,0,85.0,2.74,16.1,3,1,36276.0,France,Italy,completed


In [5]:
F['A_goals_per_match_by_edition']

In [6]:
T['A3_highest_scoring_editions'], T['A4_lowest_scoring_editions']

(      season_id  goals_per_match  matches_played
 2     COPA-2007             3.31              26
 1     COPA-2004             3.00              26
 28      WC-2026             2.96             104
 5     COPA-2016             2.84              32
 14    EURO-2020             2.78              51
 9     EURO-2000             2.74              31
 20  UNL-2024-25             2.69             188
 27      WC-2022             2.69              64,
       season_id  goals_per_match  matches_played
 3     COPA-2011             2.08              26
 13    EURO-2016             2.12              51
 8     COPA-2024             2.19              32
 24      WC-2010             2.27              64
 4     COPA-2015             2.27              26
 18  UNL-2020-21             2.28             170
 15    EURO-2024             2.29              51
 23      WC-2006             2.30              64)

In [7]:
F['A_outcome_distribution']

Knockout vs group/league: `goals90_per_match` counts regulation time only (extra-time matches with an unknown 90-minute score are excluded, never guessed).

In [8]:
T['A6_knockout_vs_group']

,competition_id,phase,matches,goals_per_match_recorded,goals90_per_match,n_goals90,draw_pct
0,COPA,group/league,176,2.477,2.477,176,26.136
1,COPA,knockout,72,2.597,3.125,48,30.556
2,EURO,group/league,204,2.402,2.402,204,25.980
3,EURO,knockout,73,2.630,2.807,57,23.288
4,UNL,group/league,716,2.471,2.471,716,25.698
5,UNL,knockout,50,2.940,3.087,46,18.000
6,WC,group/league,360,2.614,2.614,360,24.167
7,WC,knockout,128,2.594,2.352,128,19.531


In [9]:
T['A7_home_vs_away_note']

,competition_id,home_win,away_win,note
0,COPA,0.468,0.258,'home' = team listed first; most tournament matches are ...
1,EURO,0.379,0.368,'home' = team listed first; most tournament matches are ...
2,UNL,0.423,0.325,'home' = team listed first; most tournament matches are ...
3,WC,0.441,0.330,'home' = team listed first; most tournament matches are ...


## B. Team level

In [10]:
T['B1_most_successful_by_ppm_min15']

,team_name,matches,wins,draws,losses,goals_for,goals_against,clean_sheets,points,editions,win_pct,points_per_match,goals_for_pm,goals_against_pm,clean_sheet_pct,titles
2,Argentina,86,56,19,11,164,69,40,187,15,65.116,2.174,1.907,0.802,46.512,3.0
67,Spain,105,65,25,15,202,94,46,220,19,61.905,2.095,1.924,0.895,43.810,6.0
11,Brazil,83,49,20,14,155,65,37,167,16,59.036,2.012,1.867,0.783,44.578,4.0
14,Colombia,60,34,13,13,78,43,37,115,12,56.667,1.917,1.300,0.717,61.667,1.0
25,France,106,59,24,23,171,108,40,201,19,55.660,1.896,1.613,1.019,37.736,3.0
48,Netherlands,86,47,22,17,162,90,28,163,16,54.651,1.895,1.884,1.047,32.558,0.0
55,Portugal,99,52,24,23,165,93,39,180,19,52.525,1.818,1.667,0.939,39.394,3.0
51,Norway,37,20,5,12,61,43,13,65,7,54.054,1.757,1.649,1.162,35.135,0.0
30,Greece,50,26,9,15,57,42,24,87,10,52.000,1.740,1.140,0.840,48.000,1.0
8,Belgium,72,39,8,25,120,82,24,125,14,54.167,1.736,1.667,1.139,33.333,0.0


In [11]:
F['B_team_attack_vs_defence']

In [12]:
T['B3_best_defences_min15_by_ga_pm']

,team_name,matches,goals_against,goals_against_pm,clean_sheet_pct
20,Colombia,60,43,0.717,61.667
9,Belarus,28,21,0.750,53.571
13,Brazil,83,65,0.783,44.578
4,Argentina,86,69,0.802,46.512
35,Georgia,34,28,0.824,44.118
39,Greece,50,42,0.840,48.000
53,Kosovo,30,26,0.867,43.333
89,Spain,105,94,0.895,43.810
62,Morocco,16,15,0.938,37.500
74,Portugal,99,93,0.939,39.394


In [13]:
T['B8_consistency_over_editions_min4'].head(15)

,team_name,editions,mean_ppm,sd_ppm,cv
48,Ivory Coast,4,1.208,0.250,0.207
82,Senegal,4,1.296,0.380,0.293
88,South Korea,7,1.034,0.382,0.369
24,Cyprus,5,1.083,0.382,0.353
65,Nigeria,4,0.667,0.385,0.577
11,Bolivia,9,0.296,0.389,1.312
73,Poland,14,0.855,0.412,0.482
13,Brazil,16,1.948,0.433,0.222
80,Saudi Arabia,5,0.600,0.435,0.724
44,Iran,5,0.800,0.447,0.559


In [14]:
F['B_points_per_match_by_edition_top_teams']